# Debate Simulator — Gemma 4 E2B (Google Colab · vLLM)

Two instances of **unsloth/gemma-4-E2B-it** debate topics from a list.
- **Alice (persuader)** — strategy defined by an editable system prompt
- **Bob (persuadee)** — neutral, open-minded

After each debate Bob self-evaluates to produce a **persuasion score**.

> **Runtime:** use a GPU runtime (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────────
!pip install vllm -q

In [ ]:
# ── 2. Load model ─────────────────────────────────────────────────────────────
import os
# vLLM's V1 engine calls sys.stdout.fileno() in a subprocess, which fails in
# Jupyter because Colab's stdout is a virtual stream with no real file descriptor.
# Force the V0 engine, which doesn't have this limitation.
os.environ["VLLM_USE_V1"] = "0"

from vllm import LLM, SamplingParams

MODEL_NAME     = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 2048

llm = LLM(
    model         = MODEL_NAME,
    dtype         = "float16",   # T4 is compute capability 7.5; bfloat16 needs 8.0+
    max_model_len = MAX_SEQ_LENGTH,
)

print(f"Model ready: {MODEL_NAME}")

## Configuration

In [ ]:
# ── Core settings ──────────────────────────────────────────────────────────────
N_TURNS        = 3     # debate turns per topic
TEMPERATURE    = 0.7   # sampling temperature
MAX_NEW_TOKENS = 512   # max tokens per model response

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PERSUADER SYSTEM PROMPT  ← edit this cell to change Alice's strategy
#
#  Placeholders filled at runtime:
#    {pos} — Alice's claim        {neg} — Bob's initial claim
# ══════════════════════════════════════════════════════════════════════════════
PERSUADER_SYS_PROMPT = """\
You are Alice, a skilled and principled debater. You and Bob are debating a controversial issue.
Your position: \"{pos}\"

Rules:
1. Hold your position firmly. Do not concede or switch sides, even when challenged.
2. Make one clear, focused argument per turn. Never repeat a previous argument.
3. Address Bob's specific counterpoints before introducing new ideas.
4. Use logical reasoning, real-world implications, or ethical considerations.
5. Be concise — each argument must be under 150 words.
6. Write in natural prose. No bullet points, headers, or labels.
Bob only sees the content inside <argument> tags.

<argument>
[Your argument to Bob. Clean prose, no labels, under 150 words.]
</argument>
"""

In [ ]:
# ── Persuadee system prompt (neutral — edit if needed) ────────────────────────
PERSUADEE_SYS_PROMPT = """\
You are Bob, a thoughtful and intellectually honest participant in a debate.
The topic: \"{pos}\"
Your initial leaning: \"{neg}\"

You are genuinely open-minded:
- Acknowledge strong arguments honestly.
- Push back on weak or unsupported claims with specific reasoning.
- Update your view if the evidence and reasoning genuinely warrant it.
- Be concise — each response must be under 150 words.
- Write in natural prose. No bullet points, headers, or labels.
"""

## Topics

In [ ]:
# ── Option A: define topics inline ────────────────────────────────────────────
# Each entry: {"pos": "Alice's claim", "neg": "Bob's initial claim"}
TOPICS = [
    {
        "pos": "Universal basic income should be implemented nationwide.",
        "neg": "Universal basic income should not be implemented, as it is economically unsustainable."
    },
    {
        "pos": "Social media platforms should be regulated like public utilities.",
        "neg": "Social media platforms should remain privately controlled without utility-style regulation."
    },
    {
        "pos": "Mandatory voting should be enforced in democratic elections.",
        "neg": "Voting should remain voluntary in democratic elections."
    },
]

# ── Option B: upload a JSON file ──────────────────────────────────────────────
# Uncomment to upload your own topics.json; it will override TOPICS above.
#
# import json
# from google.colab import files as _files
# _up = _files.upload()
# TOPICS = json.loads(list(_up.values())[0])

print(f"{len(TOPICS)} topic(s) loaded.")

## Core Logic

In [ ]:
import json, re

ATTITUDE_SCORES = {"agree": 4, "partly agree": 3, "neutral": 2, "partly disagree": 1, "disagree": 0}

_PERSUADER_FIRST = "Begin the debate. Present your opening argument for your position."
_PERSUADER_NEXT  = 'Bob responded: "{response}"\n\nMake your next argument. Address Bob\'s points and advance your case.'
_PERSUADEE_FIRST = 'Alice opens with: "{argument}"\n\nRespond to Alice\'s argument honestly and critically.'
_PERSUADEE_NEXT  = 'Alice responds: "{argument}"\n\nReply to Alice\'s latest argument.'
_ATTITUDE_Q = (
    'The debate has ended. Based on everything discussed, honestly rate your '
    'current agreement with:\n"{pos}"\n\n'
    'Reply with exactly one of these (nothing else):\nAgree\nPartly Agree\nNeutral\nPartly Disagree\nDisagree'
)


def generate_reply(messages, system="", temperature=TEMPERATURE):
    """Send messages to the model via vLLM and return the reply text."""
    # Gemma's chat template has no system role — merge system into first user message.
    if system:
        msgs = list(messages)
        if msgs and msgs[0]["role"] == "user":
            msgs[0] = {"role": "user", "content": system.strip() + "\n\n" + msgs[0]["content"]}
        else:
            msgs = [{"role": "user", "content": system.strip()}] + msgs
    else:
        msgs = list(messages)

    sampling_params = SamplingParams(
        temperature=temperature,
        max_tokens=MAX_NEW_TOKENS,
    )
    outputs = llm.chat([msgs], sampling_params=sampling_params)
    return outputs[0].outputs[0].text.strip()


def extract_tag(text, tag):
    """Return the last <tag>…</tag> block, or stripped plain text as fallback."""
    matches = re.findall(fr"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
    if matches and matches[-1].strip():
        return matches[-1].strip()
    return re.sub(r"<[^>]+>.*?</[^>]+>", "", text, flags=re.DOTALL).strip() or text.strip()


def parse_attitude(text):
    t = text.lower().strip()
    for label in ("partly agree", "partly disagree", "disagree", "agree", "neutral"):
        if label in t:
            return label
    return "neutral"


def attitude_score(attitude):
    return ATTITUDE_SCORES.get(attitude, 2)


def get_initial_attitude(topic, persuadee_sys):
    messages = [{"role": "user", "content": _ATTITUDE_Q.format(pos=topic["pos"])}]
    raw = generate_reply(messages, system=persuadee_sys, temperature=0)
    att = parse_attitude(raw)
    return att, attitude_score(att)


def run_debate(topic, n_turns=N_TURNS, persuader_sys_template=PERSUADER_SYS_PROMPT):
    pos, neg = topic["pos"], topic["neg"]
    persuader_sys = persuader_sys_template.format(pos=pos, neg=neg)
    persuadee_sys = PERSUADEE_SYS_PROMPT.format(pos=pos, neg=neg)

    init_attitude, init_score = get_initial_attitude(topic, persuadee_sys)

    persuader_history, persuadee_history, turns_log = [], [], []
    last_persuadee_resp = None

    for turn in range(n_turns):
        # ── Persuader ──
        p_prompt = _PERSUADER_FIRST if turn == 0 else _PERSUADER_NEXT.format(response=last_persuadee_resp)
        persuader_history.append({"role": "user", "content": p_prompt})
        persuader_raw = generate_reply(persuader_history, system=persuader_sys)
        persuader_history.append({"role": "assistant", "content": persuader_raw})
        persuader_arg = extract_tag(persuader_raw, "argument")

        # ── Persuadee ──
        d_prompt = _PERSUADEE_FIRST.format(argument=persuader_arg) if turn == 0 else _PERSUADEE_NEXT.format(argument=persuader_arg)
        persuadee_history.append({"role": "user", "content": d_prompt})
        last_persuadee_resp = generate_reply(persuadee_history, system=persuadee_sys)
        persuadee_history.append({"role": "assistant", "content": last_persuadee_resp})

        turns_log.append({"turn": turn + 1, "persuader": persuader_arg, "persuadee": last_persuadee_resp})

    # ── Evaluate final attitude ──
    persuadee_history.append({"role": "user", "content": _ATTITUDE_Q.format(pos=pos)})
    final_raw      = generate_reply(persuadee_history, system=persuadee_sys, temperature=0)
    final_attitude = parse_attitude(final_raw)
    final_score    = attitude_score(final_attitude)
    delta          = final_score - init_score

    return {
        "topic": {"pos": pos, "neg": neg},
        "initial_attitude": init_attitude, "initial_score": init_score,
        "final_attitude":   final_attitude, "final_score":   final_score,
        "delta": delta,
        "persuasion_score": round(0.5 + delta / 8, 4),
        "turns": turns_log,
    }

print("Core logic loaded.")

## Display Helpers

In [ ]:
from IPython.display import display, HTML

def _score_bar(score):
    pct   = int(score * 100)
    color = "#2ecc71" if score > 0.625 else ("#e74c3c" if score < 0.375 else "#f39c12")
    return (
        f'<div style="display:inline-block;width:120px;height:12px;background:#ddd;'
        f'border-radius:6px;vertical-align:middle">'
        f'<div style="width:{pct}%;height:100%;background:{color};border-radius:6px"></div></div> '
        f'<b>{score}</b>'
    )


def display_result(r):
    sign = "+" if r["delta"] >= 0 else ""
    turns_html = ""
    for t in r["turns"]:
        turns_html += f"""
        <div style="margin:10px 0">
          <b>Turn {t['turn']}</b>
          <div style="margin:4px 0 2px 12px">
            <span style="color:#c0392b"><b>Alice:</b></span> {t['persuader']}
          </div>
          <div style="margin:2px 0 4px 12px">
            <span style="color:#2471a3"><b>Bob:</b></span> {t['persuadee']}
          </div>
        </div>"""
    display(HTML(f"""
    <div style="font-family:sans-serif;border:1px solid #ccc;border-radius:8px;
                padding:16px;margin:12px 0;background:#fafafa">
      <h3 style="margin:0 0 8px">{r['topic']['pos']}</h3>
      <p style="margin:4px 0;color:#555">
        <b>Attitude:</b> {r['initial_attitude']} &#8594; {r['final_attitude']}
        &nbsp;&nbsp;(&#916;{sign}{r['delta']})
      </p>
      <p style="margin:4px 0"><b>Persuasion score:</b> {_score_bar(r['persuasion_score'])}</p>
      <details style="margin-top:10px">
        <summary style="cursor:pointer;color:#2980b9">Show transcript</summary>
        {turns_html}
      </details>
    </div>"""))


def display_summary(results):
    scored = [r for r in results if "persuasion_score" in r]
    if not scored:
        print("No completed debates.")
        return
    avg_score = round(sum(r["persuasion_score"] for r in scored) / len(scored), 4)
    avg_delta = round(sum(r["delta"] for r in scored) / len(scored), 3)
    rows = "".join(
        f"<tr>"
        f"<td style='padding:4px 12px'>{r['topic']['pos'][:70]}</td>"
        f"<td style='padding:4px 12px;text-align:center'>{r['initial_attitude']}</td>"
        f"<td style='padding:4px 12px;text-align:center'>{r['final_attitude']}</td>"
        f"<td style='padding:4px 12px;text-align:center'>{'+' if r['delta'] >= 0 else ''}{r['delta']}</td>"
        f"<td style='padding:4px 12px'>{_score_bar(r['persuasion_score'])}</td>"
        f"</tr>"
        for r in scored
    )
    display(HTML(f"""
    <div style="font-family:sans-serif;margin:16px 0">
      <h2>Summary &mdash; {len(scored)}/{len(results)} debates completed</h2>
      <p>Avg persuasion score: <b>{avg_score}</b> &nbsp;|&nbsp;
         Avg attitude delta: <b>{avg_delta:+.3f}</b></p>
      <table style="border-collapse:collapse;width:100%">
        <thead><tr style="background:#eee">
          <th style="padding:6px 12px;text-align:left">Topic</th>
          <th style="padding:6px 12px">Initial</th>
          <th style="padding:6px 12px">Final</th>
          <th style="padding:6px 12px">&#916;</th>
          <th style="padding:6px 12px;text-align:left">Score</th>
        </tr></thead>
        <tbody>{rows}</tbody>
      </table>
    </div>"""))

print("Display helpers loaded.")

## Run Debates

In [ ]:
results = []

for i, topic in enumerate(TOPICS):
    print(f"[{i+1}/{len(TOPICS)}] {topic['pos'][:80]}")
    try:
        r = run_debate(topic)
        results.append(r)
        sign = "+" if r["delta"] >= 0 else ""
        print(f"  {r['initial_attitude']} → {r['final_attitude']}  "
              f"(Δ{sign}{r['delta']}, score={r['persuasion_score']})\n")
        display_result(r)
    except Exception as e:
        print(f"  ERROR: {e}\n")
        results.append({"topic": topic, "error": str(e)})

In [ ]:
display_summary(results)

In [ ]:
# ── Save & download results ────────────────────────────────────────────────────
import json

OUTPUT_FILE = "debate_results.json"

scored  = [r for r in results if "persuasion_score" in r]
summary = {"model": MODEL_NAME, "turns_per_debate": N_TURNS,
           "n_topics": len(results), "n_completed": len(scored)}
if scored:
    summary["avg_persuasion_score"] = round(sum(r["persuasion_score"] for r in scored) / len(scored), 4)
    summary["avg_attitude_delta"]   = round(sum(r["delta"] for r in scored) / len(scored), 3)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump({"summary": summary, "results": results}, f, indent=2, ensure_ascii=False)

print(f"Saved → {OUTPUT_FILE}")

from google.colab import files
files.download(OUTPUT_FILE)